# Lesson 02 - Preskúmanie rámca Microsoft Agent

**Microsoft Agent Framework (MAF)** je jednotný rámec na tvorbu AI agentov. Poskytuje čistú, kompozitnú architektúru so štyrmi základnými stavebnými blokmi:

- **Client** – pripája sa k endpointu AI modelu a zabezpečuje komunikáciu
- **Agent** – obaluje klienta s inštrukciami a definíciami nástrojov
- **Tools** – rozširujú schopnosti agenta o vlastné funkcie, ktoré model môže volať
- **Session** – uchováva históriu rozhovoru pre viackolové interakcie

V tejto lekcii vytvoríme **agenta na rezerváciu ciest**, ktorý pomocou týchto konceptov kontroluje dostupnosť cieľových miest.


## Inštalácia


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pochopenie architektúry rámca Agenta

Microsoft Agent Framework používa viacvrstvovú architektúru:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – `AzureAIProjectAgentProvider` sa pripája k nasadeniu Azure OpenAI. Zabezpečuje overovanie, formátovanie požiadaviek a analýzu odpovedí.
2. **Agent** – Vytvára sa z klienta pomocou `provider.create_agent()`, agent kombinuje prístup k modelu s inštrukciami (systémový prompt) a nástrojmi.
3. **Nástroje** – Python funkcie označené dekorátorom `@tool`, ktoré môže agent vyvolávať na vykonávanie akcií alebo získavanie údajov.
4. **Session** – Objekt `AgentSession` (vytvorený pomocou `agent.create_session()`), ktorý uchováva históriu konverzácie, čo umožňuje viacotáčkový dialóg, kde si agent pamätá predchádzajúci kontext.

Poďme vybudovať každú vrstvu krok za krokom.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pridávanie nástrojov s dekorátorom @tool

Nástroje umožňujú agentom vykonávať akcie nad rámec generovania textu. Dekorátor `@tool` premení obyčajnú Python funkciu na niečo, čo môže agent volať.

Kľúčové body:
- Použite `Annotated[type, "description"]`, aby model rozumel každému parametru.
- Dokumentačný reťazec sa stáva popisom nástroja, ktorý model vidí.
- `approval_mode="never_require"` znamená, že nástroj beží automaticky bez potvrdenia od používateľa.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Vytvorenie agenta s nástrojmi

Teraz skombinujeme klienta, inštrukcie a nástroje do agenta. `instructions` slúžia ako systémový prompt — definujú osobnosť a správanie agenta.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Viacnásobné rozhovory s reláciami

`AgentSession` (vytvorená pomocou `agent.create_session()`) sleduje všetky správy v konverzácii. Tým, že rovnakú reláciu odovzdáme do každého volania `agent.run()`, má agent prístup k celej histórii konverzácie a môže sa odvolávať na predchádzajúce správy.

Odovzdávame `tools=[check_destination_availability]`, aby agent mohol počas každého kola volať náš kontrolór dostupnosti.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Zhrnutie

V tejto lekcii ste spoznali štyri piliere rámca Microsoft Agent Framework:

| Koncept | Čo ste sa naučili |
|---------|--------------------|
| **Klient** | `AzureAIProjectAgentProvider` sa pripája k Azure OpenAI s autentifikáciou na základe poverení |
| **Agent** | `provider.create_agent()` spája pripojenie k modelu s inštrukciami a názvom |
| **Nástroje** | Dekorátor `@tool` sprístupňuje Python funkcie, ktoré agent môže volať |
| **Relácia** | `agent.create_session()` uchováva históriu konverzácie počas viacerých ťahov |

Tieto stavebné prvky sa spájajú, aby vytvorili agentov, ktorí dokážu viesť prirodzené konverzácie, volať externé funkcie a udržiavať kontext — základ pre zložitejšie agentné vzory v nasledujúcich lekciách.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zrieknutie sa zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, majte prosím na pamäti, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za akékoľvek nedorozumenia alebo nesprávne výklady vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
